# Patellar cartilage thickness with SKM-TEA

Run this notebook from top to bottom in a **Python 3.11** kernel. It is the project's
single entry point; reusable code lives in [`skm_tea_cdi/`](skm_tea_cdi/).

1. Configure your local data and model paths.
2. Validate DICOM, TXT, and Excel inputs before loading the model.
3. Segment six reference slices at each visit and measure cartilage thickness.
4. Inspect manual validation, longitudinal changes, and optional cohort summaries.
5. Export tables and figures under `results/`.

See [README](ReadMe.md) for installation and [data formats](docs/data-formats.md)
for the files you must supply. Copyrighted inputs and run outputs are excluded from Git.
Thickness and bone length use **pixels**. Scan totals sum thicknesses; they are not
the area-valued `CDI` field in the source spreadsheets.

## 1. Configure the run

The defaults match the local Phase 3 sample layout. Change `scan_id` to your own
identifier, or set it to `None` for the entire cohort. The default
`annotation_source="txt"` uses the six actual slices recorded in each manual
annotation, with Excel as an independent bone-length reference.

`annotation_source="excel"` reproduces the original baseline slice-range formula;
follow-up still uses the actual annotated slices. Some local baseline annotations
differ from that formula; use TXT mode to process them without guessing a match.

For TXT-only input, set both patella workbook paths to `None`. `kl_excel=None`
skips KL stratification. Baseline/follow-up Excel times are explicit and do not
have to match the numeric visit suffix. All relative paths start at the repository root.

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

from skm_tea_cdi.config import Settings

settings = Settings(
    dicom_dir=Path("datasets/dess-phase-3"),
    annotation_dir=Path("datasets/u01-phase-3"),
    medial_excel=Path("datasets/clinical/Phase_3_Medial_Patella.xls"),
    lateral_excel=Path("datasets/clinical/Phase_3_Lateral_Patella.xls"),
    kl_excel=Path("datasets/clinical/KL for Ming May 12 2016.xls"),
    model_config="models/config.yaml",
    model_weights="models/model.ckpt",
    output_dir=Path("results"),
    scan_id="9493245",  # None processes all scan folders.
    baseline_visit="v00",
    followup_visit="v03",
    baseline_time=0,
    followup_time=24,
    annotation_source="txt",
    device="auto",  # CUDA when available, otherwise CPU.
)
SAVE_FIGURES = True
RUN_CDI_ANALYSIS = True  # False runs only full volumes; manual annotations are then unnecessary.
RUN_WHOLE_VOLUMES = False  # All slices, all selected scans/visits; much slower.

## 2. Validate and inventory inputs

This checks both visits, knee identity, workbook columns, the six selected DICOM
slices, and all eight manual POI endpoints per slice. Missing or ambiguous inputs
raise an actionable error here. Each workbook is read once.

With CDI disabled, preflight checks the two image series and optional KL input
without requiring manual annotations or patella reference workbooks.

In [ ]:
from skm_tea_cdi.analysis import prepare_inputs, prepare_volume_inputs

if not RUN_CDI_ANALYSIS and not RUN_WHOLE_VOLUMES:
    raise ValueError("Enable CDI analysis, whole-volume segmentation, or both")
inputs = prepare_inputs(settings) if RUN_CDI_ANALYSIS else prepare_volume_inputs(settings)
settings.output_dir.mkdir(parents=True, exist_ok=True)
display(inputs.inventory)

## 3. Load the pretrained segmentation model

Local model files are ignored by Git. To download the published model on first use,
replace the two model paths with `MODEL_CONFIG_URL` and `MODEL_WEIGHTS_URL` from
`skm_tea_cdi.config`. Only load checkpoints from a trusted source.

The upstream library may warn that its training dataset is not registered; this
workflow supplies its own DICOM images and does not require that training dataset.

In [ ]:
from skm_tea_cdi.inference import load_model, resolve_device

model = load_model(settings.model_config, settings.model_weights, settings.device)
print(f"Model loaded on {resolve_device(settings.device)}")

## 4. Segment and measure both visits

The original preprocessing is retained: intensity normalization, CLAHE, Gaussian
blur, erosion, then z-score normalization. Label 1 is patellar cartilage. We retain
the largest connected component, trace and smooth both edges, and measure thickness
at four arc coordinates on each of six slices.

Each requested slice is inferred once per visit. Missing patellar masks produce
`NaN` measurements and `mask_found=False`; they never become zero thickness.

In [ ]:
if RUN_CDI_ANALYSIS:
    from skm_tea_cdi.analysis import analyze

    result = analyze(settings, inputs, model)
    display(result.points.head(12))
    display(result.bones)

## 5. Inspect image and manual validation

For cohorts, the figures below show the first scan only. CSV/Excel exports include
every selected scan. Cyan/magenta are calculated edges; lime is the manual bone
outline; orange marks manual thickness endpoints.

In [ ]:
if RUN_CDI_ANALYSIS:
    from skm_tea_cdi.plotting import plot_slices

    preview_scan = inputs.scan_ids[0]
    for visit in settings.visits:
        figure = plot_slices(result, preview_scan, visit)
        if SAVE_FIGURES:
            figure.savefig(settings.output_dir / f"slices_{preview_scan}_{visit}.png", dpi=120)
        display(figure)
        plt.close(figure)

## 6. Longitudinal comparison and export

Pair visits by scan, slice coordinate, and CDI location. Delta is **follow-up minus
baseline**; error is **calculated minus manual**. Per-scan totals require all 24
paired measurements. Incomplete totals remain missing. Correlations require at
least two finite, nonconstant pairs. The Bland–Altman limits use sample standard
deviation; point-level results are descriptive because measurements within a knee
are not independent.

In [ ]:
if RUN_CDI_ANALYSIS:
    from skm_tea_cdi.analysis import export_results

    tables = export_results(settings, inputs, result)
    display(tables["agreement"])
    display(tables["longitudinal_changes"])
    display(tables["scan_totals"])
    display(tables["scan_total_changes"])
    display(tables["scan_total_agreement"])
    print(f"Tables and analysis.xlsx saved under {settings.output_dir.resolve()}")

In [ ]:
if RUN_CDI_ANALYSIS:
    from skm_tea_cdi.plotting import plot_agreement, plot_kl_validation

    for visit in settings.visits:
        group = result.points[result.points.visit == visit]
        figure = plot_agreement(group.manual_px, group.calculated_px, visit)
        if SAVE_FIGURES:
            figure.savefig(settings.output_dir / f"agreement_{visit}.png", dpi=120)
        display(figure)
        plt.close(figure)

    changes = tables["longitudinal_changes"]
    figure = plot_agreement(
        changes.manual_delta_px, changes.calculated_delta_px, "Thickness change"
    )
    if SAVE_FIGURES:
        figure.savefig(settings.output_dir / "agreement_change.png", dpi=120)
    display(figure)
    plt.close(figure)

    if result.points.kl_grade.notna().any():
        display(tables["validation_by_kl"])
        figure = plot_kl_validation(result.points)
        if SAVE_FIGURES:
            figure.savefig(settings.output_dir / "displacement_by_kl.png", dpi=120)
        display(figure)
        plt.close(figure)

    # Cohort agreement uses one complete thickness total per knee, preserving the
    # original scan-level baseline, follow-up, and change analyses.
    if len(inputs.scan_ids) > 1:
        totals = tables["scan_totals"]
        for visit in settings.visits:
            group = totals[totals.visit == visit]
            figure = plot_agreement(group.manual_px, group.calculated_px, f"Scan totals {visit}")
            if SAVE_FIGURES:
                figure.savefig(settings.output_dir / f"scan_total_agreement_{visit}.png", dpi=120)
            display(figure)
            plt.close(figure)
        total_changes = tables["scan_total_changes"]
        figure = plot_agreement(
            total_changes.manual_delta_px, total_changes.calculated_delta_px, "Scan total change"
        )
        if SAVE_FIGURES:
            figure.savefig(settings.output_dir / "scan_total_agreement_change.png", dpi=120)
        display(figure)
        plt.close(figure)
        display(tables["scan_changes_by_kl"])

## 7. Optional whole-cartilage segmentation

Set `RUN_WHOLE_VOLUMES=True` in configuration to segment **all slices** in the
selected series. This replaces the former standalone whole-cartilage runner.
It uses raw-image z-score normalization, preserving that runner's preprocessing.
Labels: 0 background, 1 patellar cartilage, 2 femoral cartilage, 3 meniscus,
4 tibial cartilage. Meniscus is reported separately from cartilage.

Outputs are `volumes/<scan>/<visit>/segmentation.npy` and `volume_summary.csv` under
the configured output directory. Physical volumes are estimates from DICOM pixel
spacing and interslice spacing (or slice thickness); missing spacing yields `NaN`.

The paired volume table includes baseline/follow-up volume (cm³), absolute change,
and percent change for each tissue and whole cartilage (labels 1+2+4). KL summaries
and figures retain the former whole-cartilage runner's longitudinal analysis.
Zero baseline volume or missing physical spacing leaves percent change undefined.

For image-only analysis, set `RUN_CDI_ANALYSIS=False` and
`RUN_WHOLE_VOLUMES=True`. This indexes images directly and does not read Patella
TXT or medial/lateral workbooks. KL remains optional (`kl_excel=None` to skip).
Knee-qualified scan IDs or DICOM laterality supply knee identity for KL matching.

In [ ]:
if RUN_WHOLE_VOLUMES:
    from skm_tea_cdi.plotting import plot_volume_changes
    from skm_tea_cdi.volume import export_volumes

    volume_tables = export_volumes(settings, inputs, model)
    display(volume_tables["volume_changes"])
    display(volume_tables["volume_by_kl"])
    for name, figure in plot_volume_changes(volume_tables).items():
        if SAVE_FIGURES:
            figure.savefig(settings.output_dir / f"{name}.png", dpi=120)
        display(figure)
        plt.close(figure)
else:
    print("Whole-volume export disabled; six-slice CDI analysis is complete.")

## Finish

Inspect `mask_found`, valid-point counts, and incomplete totals before interpreting
results. Output files include local identifiers and derived measurements; keep them
private along with the source data. Before sharing changes, clear all notebook
outputs (**Edit → Clear Outputs of All Cells**) and save. The repository copy of
this notebook is intentionally unexecuted; execution evidence stays in `results/`.